Assignment — cross-sectional trend

Grid search over **bar period** (resample from source CSVs) and **lookback window tuples**; maximize **annualized Sharpe** on per-bar PnL from `run_backtest`.

### Cross-Sectional Trend

In [45]:
import sys
from itertools import product
import time
from tqdm import tqdm

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

%load_ext autoreload
%autoreload 2

sys.path.append("../../")
from src.lib.signals.cross_sectional_trend import CrossSectionalTrendEngine
from src.lib.signals.stat_arb_pca import PCAStatArbEngine

MARKET_DATA_PATH = "../../market_data/intraday/"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
TICKERS = ["SPY", "AMZN", "CVX", "GOOG", "GS", "JPM", "MS", "MSFT", "XOM"]

# Minute size embedded in filenames (e.g. SPY_5mn.csv)
SOURCE_BAR_MIN = 5

# Target bar sizes after resampling (must be >= SOURCE_BAR_MIN; use multiples for clean alignment)
PERIODS = [5, 10, 15, 20, 30, 45, 60, 90, 120, 240, 390, 1440]

# Each entry is one tuple of lookbacks (rows) passed to CrossSectionalTrendEngine
LOOKBACK_GRIDS = [
    (12, 48, 96),
    (12, 24, 48),
    (24, 48, 96),
    (12,),(24,),(36,),(48,),(60,), (72,), (96,),(120,),(144,),(168,),(192,),(240,),(360,),(480,),(720,),(1440,)
]

TOP_KS = range(1, len(TICKERS)//2)

FEE_BPS = 0.0
TRADING_HOURS = 6.5
TRADING_DAYS = 252

In [3]:
def load_close_panel(tickers: list[str], source_bar_min: int, path: str) -> pd.DataFrame:
    """Wide Close matrix at native CSV bar size."""
    frames = []
    for t in tickers:
        fp = f"{path}{t}_{source_bar_min}mn.csv"
        df = pd.read_csv(fp, index_col=0, parse_dates=True)[["Close"]]
        df.columns = [t]
        frames.append(df)
    out = pd.concat(frames, axis=1).sort_index()
    return out


def resample_close(prices: pd.DataFrame, period_min: int) -> pd.DataFrame:
    """Downsample to `period_min`-minute bars (last close)."""
    return prices.resample(f"{period_min}min").last().dropna(how="any")


def pnl_from_cumulative(cum: pd.Series) -> pd.Series:
    """Recover per-bar net PnL from cumulative sum returned by run_backtest."""
    r = cum.diff()
    r.iloc[0] = cum.iloc[0]
    return r


def annualized_sharpe(
    bar_pnl: pd.Series,
    bar_minutes: int,
    *,
    trading_hours: float = TRADING_HOURS,
    trading_days: int = TRADING_DAYS,
) -> float:
    """Sharpe on i.i.d. bar returns: sqrt(bars/year) * mean/std (ddof=1)."""
    r = bar_pnl.replace([np.inf, -np.inf], np.nan).dropna()
    if len(r) < 2 or r.std(ddof=1) == 0 or not np.isfinite(r.std(ddof=1)):
        return float("nan")
    bars_per_year = trading_days * (trading_hours * 60.0 / bar_minutes)
    return float(np.sqrt(bars_per_year) * r.mean() / r.std(ddof=1))


def run_cst_backtest(prices: pd.DataFrame, lookbacks: tuple[int, ...], top_k: int, fee_bps: float) -> pd.Series:
    cst = CrossSectionalTrendEngine(prices, lookbacks, fee_bps)
    cst.compute_signal()
    cst.generate_weights(top_k=top_k)
    return cst.run_backtest()


def grid_search_cst(
    prices_native: pd.DataFrame,
    periods: list[int],
    lookback_grids: list[tuple[int, ...]],
    top_ks: list[int],
    fee_bps: float,
    source_bar_min: int,
) -> tuple[pd.DataFrame, dict]:
    rows = []
    best = {"sharpe": -np.inf, "period": None, "lookbacks": None, "cum_rtns": None}
    for period, lbs, top_k in product(periods, lookback_grids, top_ks):
        if period < source_bar_min:
            continue
        px = resample_close(prices_native, period)
        need = max(lbs)
        if len(px) < need + 5:
            continue
        cum = run_cst_backtest(px, lbs, top_k, fee_bps)
        bar_pnl = pnl_from_cumulative(cum)
        sh = annualized_sharpe(bar_pnl, period)
        rows.append(
            {
                "period_min": period,
                "lookbacks": lbs,
                "top_k": top_k,
                "sharpe": sh,
                "cum_final": float(cum.iloc[-1]),
                "n_bars": len(px),
            }
        )
        if np.isfinite(sh) and sh > best["sharpe"]:
            best = {
                "sharpe": sh,
                "period": period,
                "lookbacks": lbs,
                "top_k": top_k,
                "cum_rtns": cum,
            }
    results = pd.DataFrame(rows)
    if not results.empty:
        results = results.sort_values("sharpe", ascending=False, na_position="last")
    return results, best

In [4]:
# load prices at native bar size (5mn)
prices_native = load_close_panel(TICKERS, SOURCE_BAR_MIN, MARKET_DATA_PATH)

In [5]:
# downsample and prepare for CST
prices = resample_close(prices_native, 60)

# plot price index
fig = px.line(prices/prices.iloc[0], title=None, template="plotly_dark")
fig.update_layout(xaxis_title=None, yaxis_title="Price index", legend_title="Ticker")
fig.show()

In [6]:
# run grid search over period, lookbacks, top_k
results, best = grid_search_cst(
    prices_native, PERIODS, LOOKBACK_GRIDS, TOP_KS, FEE_BPS, SOURCE_BAR_MIN
)
results.head(15)

,period_min,lookbacks,top_k,sharpe,cum_final,n_bars
566,240,"(720,)",3,0.830173,0.182934,1456
619,390,"(480,)",2,0.806730,0.209880,960
561,240,"(480,)",1,0.787985,0.276750,1456
279,30,"(720,)",1,0.773047,0.258523,6477
452,90,"(720,)",3,0.749649,0.152935,2493
620,390,"(480,)",3,0.731797,0.163648,960
168,15,"(1440,)",1,0.719128,0.237369,12954
341,45,"(1440,)",3,0.661373,0.126089,4485
486,120,"(120,)",1,0.630381,0.218037,1992
426,90,"(96,)",1,0.621128,0.206070,2493


In [7]:
# show optimal strategy (max Sharpe)
fig = px.line(
    best["cum_rtns"],
    title=(
        f"Best cumulative PnL — period={best['period']}min, "
        f"lookbacks={best['lookbacks']}, top_k={best['top_k']}, Sharpe={best['sharpe']:.3f}"
    ),
)
fig.update_layout(xaxis_title=None, yaxis_title="Cumulative (sum of bar PnL)", showlegend=False)
fig.show()

In [8]:
top_k = best["top_k"]

# filter results by top_k
results_plot = results[results["top_k"] == top_k]

pivot = results_plot.pivot_table(
    index="lookbacks", columns="period_min", values="sharpe", aggfunc="first"
)
fig = go.Figure(
    data=go.Heatmap(
        z=pivot.to_numpy(),
        x=pivot.columns.astype(str),
        y=pivot.index.astype(str),
        colorscale="RdYlGn",
        colorbar=dict(title="Sharpe"),
    )
)
fig.update_layout(
    title=f"Annualized Sharpe: period (columns) × lookbacks (rows) for top_k = {top_k}",
    xaxis_title="Bar period (minutes)",
    yaxis_title="Lookback tuple",
)
fig.show()

### Statistical Arbitrage

In [37]:
# plot PCA explained variance
prices = resample_close(prices_native, 60)
n_comp = len(TICKERS)    # use all components
lookback_window = len(prices)    # use all bars

pca_engine = PCAStatArbEngine(prices, n_components=n_comp, lookup_window=lookback_window)

pca_decomp, _ = pca_engine._get_pca(prices)

variance_ratios = pca_decomp.explained_variance_ratio_
cumulative_variance = variance_ratios.cumsum()

In [ ]:
# variance_ratios → left y-axis (bars); cumulative_variance → right y-axis (line)
_pc_x = [f"PC{i + 1}" for i in range(len(variance_ratios))]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=_pc_x, y=variance_ratios, name="Explained variance ratio"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=_pc_x,
        y=cumulative_variance,
        name="Cumulative explained variance",
        mode="lines+markers",
    ),
    secondary_y=True,
)
fig.update_layout(
    xaxis_title="Principal component",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
)
fig.update_yaxes(
    title_text="Explained variance ratio",
    secondary_y=False,
    tickformat=".0%",
    showgrid=False,
)
fig.update_yaxes(
    title_text="Cumulative explained variance",
    secondary_y=True,
    tickformat=".0%",
    showgrid=False,
)
fig.show()

In [15]:
def run_pca_backtest(prices: pd.DataFrame, period: int, n_comp: int, lookback_window: int, score_threshold: float) -> pd.Series:
    pca_engine = PCAStatArbEngine(prices, n_components=n_comp, lookup_window=lookback_window)
    scores = pca_engine.generate_signals()
    weights = pca_engine.compute_weights(scores, score_threshold)
    pnl_stats, cum_rets = pca_engine.calculate_performance(weights, bars_per_day=6.5*60/period)
    return pnl_stats, cum_rets

In [48]:
# hyperparameter tuning
PERIODS = [60, 90, 120]
PCA_COMPONENTS = [2, 3]
PCA_WINDOWS = [240, 480, 600, 720]
SCORE_THRESHOLDS = np.arange(1.5, 3.1, 0.25)

# optimize for best PnL or best Sharpe
BEST_PNL = True

best_sharpe = -np.inf
best_pnl = -np.inf
best_params = None

for period in PERIODS:
    # downsample and prepare for PCA
    prices = resample_close(prices_native, period)
    for n_comp, lookback_window in tqdm(product(PCA_COMPONENTS, PCA_WINDOWS)):
        pca_engine = PCAStatArbEngine(prices, n_components=n_comp, lookup_window=lookback_window)
        scores = pca_engine.generate_signals()
        for score_threshold in SCORE_THRESHOLDS:
            tic = time.time()
            weights = pca_engine.compute_weights(scores, score_threshold)
            pnl_stats, cum_rets = pca_engine.calculate_performance(weights, bars_per_day=6.5*60/period)
            toc = time.time()
            # print(f"Period/PCA/Score {period}/{n_comp}/{lookback_window}/{score_threshold}: {round(toc - tic, 2)}s")
            if (BEST_PNL and pnl_stats["Total Return"] > best_pnl) or (not BEST_PNL and pnl_stats["Annualized Sharpe"] > best_sharpe):
                best_pnl = pnl_stats["Total Return"]
                best_sharpe = pnl_stats["Annualized Sharpe"]
                best_params = {"period": period, "n_comp": n_comp, "lookback_window": lookback_window, "score_threshold": round(score_threshold, 2)}

print(f"Best PNL/Sharpe: {best_pnl:.1f}/{best_sharpe:.3f} with params: {best_params}")

0it [00:00, ?it/s]

8it [00:17,  2.24s/it]
8it [00:12,  1.51s/it]
8it [00:09,  1.17s/it]

Best PNL/Sharpe: 0.3/1.950 with params: {'period': 90, 'n_comp': 3, 'lookback_window': 480, 'score_threshold': np.float64(2.5)}


In [49]:
# run with best params
period = best_params["period"]
prices = resample_close(prices_native, period)
pnl_stats, cum_rets = run_pca_backtest(prices, **best_params)

In [50]:
# plot cumulative PnL
fig = px.line(cum_rets, title="Cumulative PnL", template="plotly_dark")
fig.update_layout(xaxis_title=None, yaxis_title="Cumulative PnL", showlegend=False)
fig.show()
